# 02 - Shared Training and Validation Split

## Objective

The labelled catalogue is divided into training data and validation data. Text and image models will both use this same saved split. The official test catalogue remains separate.

## 1. Load the Labelled Products

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_utils import get_data_paths, load_merged_data

data_paths = get_data_paths()
products = load_merged_data().rename_axis("row_id").reset_index()

print(f"Labelled products: {len(products):,}")
print(f"Product classes: {products['prdtypecode'].nunique()}")

Labelled products: 84,916
Product classes: 27


## 2. Divide the Labelled Products

Eighty percent is used for training and twenty percent for validation. Stratification keeps the proportions of the 27 classes similar in both groups. The fixed random state makes the result reproducible.

In [2]:
training_rows, validation_rows = train_test_split(
    products,
    test_size=0.20,
    random_state=42,
    stratify=products["prdtypecode"],
)

## 3. Record Each Product's Group

In [3]:
training_split = training_rows[["row_id"]].copy()
training_split["split"] = "train"

validation_split = validation_rows[["row_id"]].copy()
validation_split["split"] = "validation"

split_manifest = pd.concat(
    [training_split, validation_split],
    ignore_index=True,
)

split_manifest["split"].value_counts().to_frame("products")

,products
split,
train,67932
validation,16984


## 4. Check the Stratification

The table below shows how many products from each class went into training and validation. The validation percentage should be close to 20% for every class.

In [4]:
split_check = products[["row_id", "prdtypecode"]].merge(
    split_manifest, on="row_id"
)

class_split = pd.crosstab(
    split_check["prdtypecode"],
    split_check["split"],
)
class_split["total"] = class_split["train"] + class_split["validation"]
class_split["validation %"] = (
    100 * class_split["validation"] / class_split["total"]
).round(1)

class_split

split,train,validation,total,validation %
prdtypecode,,,,
10,2493,623,3116,20.0
40,2006,502,2508,20.0
50,1345,336,1681,20.0
60,666,166,832,20.0
1140,2137,534,2671,20.0
1160,3162,791,3953,20.0
1180,611,153,764,20.0
1280,3896,974,4870,20.0
1281,1656,414,2070,20.0


## 5. Check and Save the Shared Split

Every labelled product must appear exactly once. The saved file contains only its `row_id` and its assigned group.

In [5]:
assert len(split_manifest) == len(products)
assert split_manifest["row_id"].is_unique

split_path = Path(data_paths["processed"]) / "data_split.csv"
split_path.parent.mkdir(parents=True, exist_ok=True)
split_manifest.to_csv(split_path, index=False)

print(f"Saved shared split: {split_path}")

Saved shared split: /Users/wahajaslam/capstone_raukaten/data/processed/data_split.csv


## Conclusion

The same `data_split.csv` file will be joined to text and image data using `row_id`. This keeps both model evaluations synchronized.